# Business Analytics – Denormalised Gold Table

This notebook contains the Athena CTAS query that combines the Gold fact table with date and product attributes.

**Output table:** `retail_gold_db.gld_business_analytics`  
**S3 destination:** `s3://tus-retail-clothing-store-datasets/gold/business_analytics/`

The query preserves the original Databricks join pattern: fact → date and fact → product. It does not join `dim_customers`.

In [ ]:
-- Create the denormalised business analytics table
CREATE TABLE retail_gold_db.gld_business_analytics
WITH (
    format = 'PARQUET',
    external_location = 's3://tus-retail-clothing-store-datasets/gold/business_analytics/',
    write_compression = 'SNAPPY'
)
AS
SELECT
    i.*,
    c.year,
    c.month_name,
    c.day_name,
    c.is_weekend,
    c.quarter,
    c.week,
    p.sku,
    p.category_code,
    p.category_name,
    p.brand_code,
    p.brand_name,
    p.color,
    p.size,
    p.rating_count,
    EXTRACT(HOUR FROM i.transaction_ts) AS hour_of_day
FROM retail_gold_db.fact_order_items i
INNER JOIN retail_gold_db.dim_date c
    ON i.date_id = c.date_id
INNER JOIN retail_gold_db.dim_products p
    ON i.product_id = p.product_id;

## Before rerunning the CTAS

Athena CTAS does not overwrite an existing table or clean its S3 destination automatically. If you need to recreate this table, first drop the catalog table and ensure the target S3 prefix is empty. Do not delete the separate `athena-results/` prefix.

In [ ]:
-- Run only when intentionally removing the existing catalog table
DROP TABLE IF EXISTS retail_gold_db.gld_business_analytics;